In [ ]:
import torch
import IPython.display as ipd
assert torch.cuda.is_available()
%cd /mlx/users/zongyu.yin/playground/samantha

sample_rate = 16000
batch_size = 2
shuffle_buffer_size = 10
num_workers = 2
split = "small"

In [ ]:
import librosa
import matplotlib.pyplot as plt
from librosa.feature.inverse import mel_to_audio
from torchaudio.functional import DB_to_amplitude

def plot_waveform(waveform, sr, title="Waveform", ax=None):
    waveform = waveform.numpy()

    num_channels, num_frames = waveform.shape
    time_axis = torch.arange(0, num_frames) / sr

    if ax is None:
        _, ax = plt.subplots(num_channels, 1)
    ax.plot(time_axis, waveform[0], linewidth=1)
    ax.grid(True)
    ax.set_xlim([0, time_axis[-1]])
    ax.set_title(title)


def plot_spectrogram(specgram, title=None, ylabel="freq_bin", ax=None):
    if ax is None:
        _, ax = plt.subplots(1, 1)
    if title is not None:
        ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.imshow(specgram, origin="lower", aspect="auto", interpolation="nearest")

def mel2audio(mel):
    mel_db2amp = DB_to_amplitude(mel, ref=1, power=1)
    audio = mel_to_audio(mel_db2amp.numpy(), sr=sample_rate, n_fft=512, win_length=400, hop_length=160, n_iter=100)
    return audio
    

# Dataloader

In [ ]:
from recipes.datasets.librilight import LibriLightWebDataModule

pl_datamodule = LibriLightWebDataModule(
    sample_rate=sample_rate,
    split=split,
    batch_size=batch_size,
    shuffle_buffer_size=shuffle_buffer_size,
    num_workers=num_workers,
    pin_memory=True,
)

train_loader = pl_datamodule.val_dataloader()

In [ ]:
it = iter(train_loader)

In [ ]:
batch = next(it)

In [ ]:
for audio in batch["audio"]:
    ipd.display(ipd.Audio(audio, rate=sample_rate))

# Without VQ

## Load ckpt and model

In [ ]:
import os
import samantha.utils.hdfs_helper as hh
from recipes.umm.modules.lit_module import BestRQMel

ckpt_path = "hdfs://haruna/home/byte_speech_sv/zongyu.yin/logs/umm/finetune_speech_melrecon_25hz_8x4096/checkpoints/step=050000-tr_loss=3.5555-val_loss_0=3.4967.ckpt"
cache_dir = ".module_cache/umm"
local_path = f"{cache_dir}/{os.path.basename(ckpt_path)}"
if not os.path.exists(local_path):
    if not hh.get(ckpt_path, local_path):
        raise ConnectionError(f"Cannot retrieve file from {ckpt_path}.")
else:
    print("Cached ckpt found.")
bestrq_mel = BestRQMel.load_from_checkpoint(local_path).to("cuda").eval()

## Get melspec

In [ ]:
model_input = {"audio": batch["audio"].to("cuda")}
recon_mel, masked_mel, gt_mel = bestrq_mel.get_mel(model_input)
mean = bestrq_mel.model.model_input_transform.mean
std = bestrq_mel.model.model_input_transform.std
recon_mel = (recon_mel.detach().transpose(1, 2) * std + mean).transpose(1, 2).cpu()
masked_mel = (masked_mel.detach().transpose(1, 2) * std + mean).transpose(1, 2).cpu()
gt_mel = (gt_mel.detach().transpose(1, 2) * std + mean).transpose(1, 2).cpu()

## Get audio and plot

In [ ]:
for audio, recon_m, masked_m, gt_m in zip(batch["audio"], recon_mel, masked_mel, gt_mel):
    print("-----------------------")
    recon_audio = mel2audio(recon_m)
    gt_audio = mel2audio(gt_m)
    masked_audio = mel2audio(masked_m)
    fig, axs = plt.subplots(4, 1)
    print("Groud Truth")
    ipd.display(ipd.Audio(audio, rate=sample_rate))
    print("Groud Truth (mel -> audio)")
    ipd.display(ipd.Audio(gt_audio, rate=sample_rate))
    print("Masked (mel -> audio)")
    ipd.display(ipd.Audio(masked_audio, rate=sample_rate))
    print("Reconstruction (mel -> audio)")
    ipd.display(ipd.Audio(recon_audio, rate=sample_rate))
    plot_waveform(audio, sample_rate, title="Original waveform", ax=axs[0])
    plot_spectrogram(gt_m, title="GT", ax=axs[1])
    plot_spectrogram(masked_m, title="Masked", ax=axs[2])
    plot_spectrogram(recon_m, title="Recon", ax=axs[3])
    fig.tight_layout()

# With VQ

## Load ckpt and model

In [ ]:
import os
import samantha.utils.hdfs_helper as hh

ckpt_path = "hdfs://haruna/home/byte_speech_sv/zongyu.yin/logs/umm/finetune_speech_melrecon_25hz_rq8x4096_vq32768x256/checkpoints/step=030000-tr_loss=4.2303-val_loss_0=4.2687.ckpt"
cache_dir = ".module_cache/umm"
local_path = f"{cache_dir}/{os.path.basename(ckpt_path)}"
if not os.path.exists(local_path):
    if not hh.get(ckpt_path, local_path):
        raise ConnectionError(f"Cannot retrieve file from {ckpt_path}.")
else:
    print("Cached ckpt found.")
bestrq_mel_vq = BestRQMel.load_from_checkpoint(local_path).to("cuda").eval()

## Get melspec

In [ ]:
model_input = {"audio": batch["audio"].to("cuda")}
recon_mel, masked_mel, gt_mel = bestrq_mel_vq.get_mel(model_input)
mean = bestrq_mel_vq.model.model_input_transform.mean
std = bestrq_mel_vq.model.model_input_transform.std
recon_mel = (recon_mel.detach().transpose(1, 2) * std + mean).transpose(1, 2).cpu()
masked_mel = (masked_mel.detach().transpose(1, 2) * std + mean).transpose(1, 2).cpu()
gt_mel = (gt_mel.detach().transpose(1, 2) * std + mean).transpose(1, 2).cpu()

## Get audio and plot

In [ ]:
for audio, recon_m, masked_m, gt_m in zip(batch["audio"], recon_mel, masked_mel, gt_mel):
    print("-----------------------")
    recon_audio = mel2audio(recon_m)
    gt_audio = mel2audio(gt_m)
    masked_audio = mel2audio(masked_m)
    fig, axs = plt.subplots(4, 1)
    print("Groud Truth")
    ipd.display(ipd.Audio(audio, rate=sample_rate))
    print("Groud Truth (mel -> audio)")
    ipd.display(ipd.Audio(gt_audio, rate=sample_rate))
    print("Masked (mel -> audio)")
    ipd.display(ipd.Audio(masked_audio, rate=sample_rate))
    print("Reconstruction (mel -> audio)")
    ipd.display(ipd.Audio(recon_audio, rate=sample_rate))
    plot_waveform(audio, sample_rate, title="Original waveform", ax=axs[0])
    plot_spectrogram(gt_m, title="GT", ax=axs[1])
    plot_spectrogram(masked_m, title="Masked", ax=axs[2])
    plot_spectrogram(recon_m, title="Recon", ax=axs[3])
    fig.tight_layout()